<a href="https://colab.research.google.com/github/yhou151209/541_project/blob/main/541.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ortools

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
from __future__ import annotations

import json
import difflib
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


# Preference penalty bounds used in soft constraints.
MIN_PREFERENCE_PENALTY = -10
MAX_PREFERENCE_PENALTY = 10

# Accepted day aliases for flexible user input.
DAY_ALIASES = {
    "m": "Monday",
    "mon": "Monday",
    "monday": "Monday",
    "tu": "Tuesday",
    "tue": "Tuesday",
    "tues": "Tuesday",
    "tuesday": "Tuesday",
    "w": "Wednesday",
    "wed": "Wednesday",
    "weds": "Wednesday",
    "wednesday": "Wednesday",
    "th": "Thursday",
    "thu": "Thursday",
    "thur": "Thursday",
    "thurs": "Thursday",
    "thursday": "Thursday",
    "f": "Friday",
    "fri": "Friday",
    "friday": "Friday",
    "sa": "Saturday",
    "sat": "Saturday",
    "saturday": "Saturday",
    "su": "Sunday",
    "sun": "Sunday",
    "sunday": "Sunday",
}

# Accepted shift aliases for flexible user input.
SHIFT_ALIASES = {
    "morning": "morning",
    "am": "morning",
    "a m": "morning",
    "a.m": "morning",
    "a.m.": "morning",
    "day": "morning",
    "dayshift": "morning",
    "day shift": "morning",
    "evening": "evening",
    "pm": "evening",
    "p m": "evening",
    "p.m": "evening",
    "p.m.": "evening",
    "night": "evening",
    "nightshift": "evening",
    "night shift": "evening",
}

# Accepted yes/no aliases for flexible prompts.
YES_ALIASES = {"y", "yes", "yeah", "yep", "sure", "ok", "okay", "true", "1"}
NO_ALIASES = {"n", "no", "nope", "false", "0"}

# Fixed weekday order used across sorting and adjacency logic.
ORDERED_DAYS = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]


# ---------------------------
# Text normalization utilities
# ---------------------------

def simplify_text(value: str) -> str:
    """Normalize free-text input for robust matching."""
    cleaned = value.strip().lower()
    cleaned = cleaned.replace("_", " ")
    cleaned = re.sub(r"[.]+", "", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned


def resolve_with_aliases(
    raw_value: str,
    aliases: Dict[str, str],
    label: str,
    examples: str,
    cutoff: float = 0.72,
) -> str:
    """Resolve user input through aliases and light fuzzy matching."""
    normalized = simplify_text(raw_value)
    compact = normalized.replace(" ", "")

    if normalized in aliases:
        return aliases[normalized]
    if compact in aliases:
        return aliases[compact]

    candidates = list(
        dict.fromkeys(list(aliases.keys()) + [k.replace(" ", "") for k in aliases.keys()])
    )

    match = difflib.get_close_matches(normalized, candidates, n=1, cutoff=cutoff)
    if not match:
        match = difflib.get_close_matches(compact, candidates, n=1, cutoff=cutoff)

    if match:
        matched_key = match[0]
        if matched_key in aliases:
            return aliases[matched_key]
        for key, value in aliases.items():
            if key.replace(" ", "") == matched_key:
                return value

    raise ValueError(f"Invalid {label}. Examples: {examples}.")


def normalize_day(day: str) -> str:
    """Normalize a day string to canonical weekday form."""
    return resolve_with_aliases(
        day,
        DAY_ALIASES,
        "day",
        "Mon, Tues, Wednesday, Thurs, Fri, Saturday, Sunday",
    )


def normalize_shift(shift: str) -> str:
    """Normalize a shift string to canonical shift form."""
    return resolve_with_aliases(
        shift,
        SHIFT_ALIASES,
        "shift",
        "AM, morning, day shift, PM, evening, night shift",
    )


def normalize_optional_day(day: str | None) -> str | None:
    """Normalize an optional day value."""
    if day is None:
        return None
    return normalize_day(day)


def normalize_optional_shift(shift: str | None) -> str | None:
    """Normalize an optional shift value."""
    if shift is None:
        return None
    return normalize_shift(shift)


def ask_yes_no(prompt: str) -> bool:
    """Ask a yes/no question with tolerant input parsing."""
    while True:
        answer = simplify_text(input(prompt))
        compact = answer.replace(" ", "")
        if answer in YES_ALIASES or compact in YES_ALIASES:
            return True
        if answer in NO_ALIASES or compact in NO_ALIASES:
            return False
        print("Please enter yes/y or no/n.")


def resolve_employee_name(staff: List[Dict[str, Any]], employee_name: str) -> Dict[str, Any]:
    """Resolve an employee name with case-insensitive and fuzzy matching."""
    normalized = simplify_text(employee_name)
    name_lookup = {simplify_text(s["name"]): s for s in staff}

    if normalized in name_lookup:
        return name_lookup[normalized]

    matches = difflib.get_close_matches(normalized, list(name_lookup.keys()), n=1, cutoff=0.72)
    if matches:
        return name_lookup[matches[0]]

    available_names = ", ".join(s["name"] for s in staff)
    raise ValueError(
        f"Employee '{employee_name}' not found. Available employees: {available_names}."
    )


# ---------------------------
# Data validation and IO
# ---------------------------

def clamp_preference_penalty(value: int) -> int:
    """Validate that a preference penalty is within allowed bounds."""
    if value < MIN_PREFERENCE_PENALTY or value > MAX_PREFERENCE_PENALTY:
        raise ValueError(
            f"Penalty must be between {MIN_PREFERENCE_PENALTY} and {MAX_PREFERENCE_PENALTY}."
        )
    return value


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    """Load and validate scheduling data from a JSON file."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    """Save scheduling data back to a JSON file."""
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    """Validate required schema and normalize day/shift fields."""
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts", "employment_type"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")
        row["day"] = normalize_day(row["day"])
        row["shift"] = normalize_shift(row["shift"])

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")
        req["day"] = normalize_day(req["day"])
        req["shift"] = normalize_shift(req["shift"])

    if "preferences" in data:
        for pref in data["preferences"]:
            if "type" not in pref or "employee_name" not in pref:
                raise ValueError(f"Preference missing required fields: {pref}")

            if "day" in pref:
                pref["day"] = normalize_day(pref["day"])
            if "shift" in pref:
                pref["shift"] = normalize_shift(pref["shift"])

            if pref["type"] in {"shift_preference", "avoid_shift"}:
                penalty = int(pref.get("penalty", 5))
                clamp_preference_penalty(penalty)

            if pref["type"] in {"back_to_back_preference", "avoid_back_to_back"}:
                penalty = int(pref.get("penalty", 5))
                clamp_preference_penalty(penalty)


def clone_data(data: Dict[str, Any]) -> Dict[str, Any]:
    """Create a deep-ish copy of the scheduling data structure."""
    return {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        "preferences": [dict(p) for p in data.get("preferences", [])],
    }


def clone_schedule(schedule: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """Clone a schedule list."""
    return [dict(row) for row in schedule]


# ---------------------------
# Lookup and helper utilities
# ---------------------------

def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    """Build a fast lookup for employee availability."""
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """Map employee_id to staff record."""
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    """Convert a schedule into a set of assignment tuples."""
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_day_order() -> Dict[str, int]:
    """Return weekday order mapping for sorting."""
    return {day: idx + 1 for idx, day in enumerate(ORDERED_DAYS)}


def get_sorted_days(days: List[str]) -> List[str]:
    """Sort unique days according to weekday order."""
    day_order = get_day_order()
    return sorted(set(days), key=lambda d: day_order.get(d, 999))


def is_busy_shift(day: str, shift: str) -> bool:
    """Return whether a shift is considered busy."""
    normalized_day = normalize_day(day)
    normalized_shift = normalize_shift(shift)

    if normalized_shift == "evening":
        return True
    if normalized_day in {"Saturday", "Sunday"} and normalized_shift in {"morning", "evening"}:
        return True
    return False


def build_assignment_flag(
    model: cp_model.CpModel,
    vars_for_assignment: List[cp_model.IntVar],
    name: str,
) -> cp_model.IntVar:
    """Create a boolean flag indicating whether any assignment var is active."""
    flag = model.NewBoolVar(name)
    model.Add(sum(vars_for_assignment) >= 1).OnlyEnforceIf(flag)
    model.Add(sum(vars_for_assignment) == 0).OnlyEnforceIf(flag.Not())
    return flag


# ---------------------------
# Schedule query helpers
# ---------------------------

def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    """Return assignments for one employee on one specific day/shift."""
    normalized_name = simplify_text(employee_name)
    normalized_day = normalize_day(day)
    normalized_shift = normalize_shift(shift)

    matches = []
    for row in schedule:
        if (
            simplify_text(row["employee_name"]) == normalized_name
            and row["day"] == normalized_day
            and row["shift"] == normalized_shift
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    """Return assigned roles for one employee on one specific day/shift."""
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def count_matching_assignments(
    schedule: List[Dict[str, str]],
    employee_name: str,
    target_day: str | None,
    target_shift: str | None,
) -> int:
    """Count assignments matching an employee and optional day/shift pattern."""
    normalized_name = simplify_text(employee_name)
    count = 0

    for row in schedule:
        if simplify_text(row["employee_name"]) != normalized_name:
            continue
        if target_day is not None and row["day"] != target_day:
            continue
        if target_shift is not None and row["shift"] != target_shift:
            continue
        count += 1

    return count


def get_removed_and_added_assignments(
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    """Return detailed removed and added assignments between two schedules."""
    old_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in old_schedule
    }
    new_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in new_schedule
    }

    removed = old_set - new_set
    added = new_set - old_set

    removed_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(removed)
    ]

    added_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(added)
    ]

    return removed_rows, added_rows


def get_back_to_back_counts(
    schedule: List[Dict[str, str]],
    employee_name: str,
) -> Tuple[int, int]:
    """Count same-day and true cross-day back-to-back assignments."""
    normalized_name = simplify_text(employee_name)

    same_day_count = 0
    cross_day_count = 0

    for day in ORDERED_DAYS:
        has_morning = any(
            simplify_text(r["employee_name"]) == normalized_name
            and r["day"] == day
            and r["shift"] == "morning"
            for r in schedule
        )
        has_evening = any(
            simplify_text(r["employee_name"]) == normalized_name
            and r["day"] == day
            and r["shift"] == "evening"
            for r in schedule
        )
        if has_morning and has_evening:
            same_day_count += 1

    for idx in range(len(ORDERED_DAYS) - 1):
        day = ORDERED_DAYS[idx]
        next_day = ORDERED_DAYS[idx + 1]

        has_evening = any(
            simplify_text(r["employee_name"]) == normalized_name
            and r["day"] == day
            and r["shift"] == "evening"
            for r in schedule
        )
        has_next_morning = any(
            simplify_text(r["employee_name"]) == normalized_name
            and r["day"] == next_day
            and r["shift"] == "morning"
            for r in schedule
        )
        if has_evening and has_next_morning:
            cross_day_count += 1

    return same_day_count, cross_day_count


# ---------------------------
# Data mutation helpers
# ---------------------------

def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    """Set one employee's availability for one exact day/shift."""
    normalized_day = normalize_day(day)
    normalized_shift = normalize_shift(shift)

    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"] == normalized_day
            and row["shift"] == normalized_shift
        ):
            row["available"] = available
            row["day"] = normalized_day
            row["shift"] = normalized_shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": normalized_day,
                "shift": normalized_shift,
                "available": available,
            }
        )


def set_availability_by_pattern(
    data: Dict[str, Any],
    employee_id: str,
    available: bool,
    target_day: str | None = None,
    target_shift: str | None = None,
) -> None:
    """Set availability for a day-only, shift-only, or day+shift pattern."""
    target_day = normalize_optional_day(target_day)
    target_shift = normalize_optional_shift(target_shift)

    if target_day is None and target_shift is None:
        raise ValueError("At least one of target_day or target_shift must be provided.")

    all_day_shift_pairs = {
        (row["day"], row["shift"])
        for row in data["availability"]
        if row["employee_id"] == employee_id
    }
    all_day_shift_pairs.update(
        (req["day"], req["shift"])
        for req in data["shift_requirements"]
    )

    matched_pairs = []
    for day, shift in sorted(all_day_shift_pairs):
        if target_day is not None and day != target_day:
            continue
        if target_shift is not None and shift != target_shift:
            continue
        matched_pairs.append((day, shift))

    if not matched_pairs:
        raise ValueError(
            f"No matching availability slots found for employee_id={employee_id}, "
            f"day={target_day}, shift={target_shift}."
        )

    for day, shift in matched_pairs:
        set_availability(data, employee_id, day, shift, available)


def add_preference(data: Dict[str, Any], preference: Dict[str, Any]) -> None:
    """Add a normalized preference record into the data object."""
    if "day" in preference:
        preference["day"] = normalize_day(preference["day"])
    if "shift" in preference:
        preference["shift"] = normalize_shift(preference["shift"])

    if "preferences" not in data:
        data["preferences"] = []
    data["preferences"].append(preference)


# ---------------------------
# Solver
# ---------------------------

def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    """Solve the weekly schedule with hard and soft constraints."""
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]
    preferences = data.get("preferences", [])

    staff_lookup = get_staff_lookup(staff)
    name_to_id = {simplify_text(s["name"]): s["id"] for s in staff}
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})
    all_days = get_sorted_days([r["day"] for r in requirements])
    all_shifts = sorted({r["shift"] for r in requirements})

    # Create assignment variables and disable impossible assignments.
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    # Each required role count must be exactly satisfied.
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    # Each employee can hold at most one role per shift.
    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    # Respect maximum shift count per employee.
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    # Force selected assignments when required, e.g. direct swap.
    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    # Track max workload for simple balancing.
    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    # Reward keeping previous assignments when possible.
    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    # Build soft-penalty terms from preferences.
    penalty_terms: List[cp_model.LinearExpr] = []

    for i, pref in enumerate(preferences):
        pref_type = pref["type"]
        employee_name = simplify_text(pref["employee_name"])

        if employee_name not in name_to_id:
            continue

        emp_id = name_to_id[employee_name]

        if pref_type in {"shift_preference", "avoid_shift"}:
            penalty = clamp_preference_penalty(int(pref.get("penalty", 5)))
            target_day = pref.get("day")
            target_shift = pref.get("shift")

            for role in all_roles:
                for day in all_days:
                    for shift in all_shifts:
                        day_match = (target_day is None) or (day == target_day)
                        shift_match = (target_shift is None) or (shift == target_shift)

                        if day_match and shift_match:
                            key = (emp_id, day, shift, role)
                            if key in x:
                                penalty_terms.append(x[key] * penalty)

        elif pref_type in {"back_to_back_preference", "avoid_back_to_back"}:
            penalty = clamp_preference_penalty(int(pref.get("penalty", 5)))

            # Same-day morning + evening.
            for day in all_days:
                morning_vars = [
                    x[(emp_id, day, "morning", role)]
                    for role in all_roles
                    if (emp_id, day, "morning", role) in x
                ]
                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]

                if not morning_vars or not evening_vars:
                    continue

                morning_assigned = build_assignment_flag(
                    model,
                    morning_vars,
                    f"morning_assigned_{emp_id}_{day}_{i}",
                )
                evening_assigned = build_assignment_flag(
                    model,
                    evening_vars,
                    f"evening_assigned_{emp_id}_{day}_{i}",
                )
                same_day_back_to_back = model.NewBoolVar(
                    f"same_day_back_to_back_{emp_id}_{day}_{i}"
                )

                model.AddBoolAnd([morning_assigned, evening_assigned]).OnlyEnforceIf(
                    same_day_back_to_back
                )
                model.AddBoolOr(
                    [morning_assigned.Not(), evening_assigned.Not()]
                ).OnlyEnforceIf(same_day_back_to_back.Not())

                penalty_terms.append(same_day_back_to_back * penalty)

            # True cross-day evening -> next-morning.
            for day_index in range(len(all_days) - 1):
                day = all_days[day_index]
                next_day = all_days[day_index + 1]

                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]
                next_morning_vars = [
                    x[(emp_id, next_day, "morning", role)]
                    for role in all_roles
                    if (emp_id, next_day, "morning", role) in x
                ]

                if not evening_vars or not next_morning_vars:
                    continue

                evening_assigned = build_assignment_flag(
                    model,
                    evening_vars,
                    f"cross_evening_assigned_{emp_id}_{day}_{i}",
                )
                next_morning_assigned = build_assignment_flag(
                    model,
                    next_morning_vars,
                    f"cross_next_morning_assigned_{emp_id}_{next_day}_{i}",
                )
                cross_day_back_to_back = model.NewBoolVar(
                    f"cross_day_back_to_back_{emp_id}_{day}_to_{next_day}_{i}"
                )

                model.AddBoolAnd(
                    [evening_assigned, next_morning_assigned]
                ).OnlyEnforceIf(cross_day_back_to_back)
                model.AddBoolOr(
                    [evening_assigned.Not(), next_morning_assigned.Not()]
                ).OnlyEnforceIf(cross_day_back_to_back.Not())

                penalty_terms.append(cross_day_back_to_back * penalty)

    # Prefer full-time staff on busy shifts when possible.
    busy_shift_penalty_terms: List[cp_model.LinearExpr] = []
    busy_penalty_weight = 4

    for (emp_id, day, shift, role), var in x.items():
        emp_type = staff_lookup[emp_id]["employment_type"]
        if is_busy_shift(day, shift) and emp_type != "full-time":
            busy_shift_penalty_terms.append(var * busy_penalty_weight)

    all_penalties = penalty_terms + busy_shift_penalty_terms

    max_possible_preference_penalty = len(all_penalties) * MAX_PREFERENCE_PENALTY
    min_possible_preference_penalty = len(all_penalties) * MIN_PREFERENCE_PENALTY

    total_penalty = model.NewIntVar(
        min_possible_preference_penalty,
        max_possible_preference_penalty,
        "total_penalty",
    )

    if all_penalties:
        model.Add(total_penalty == sum(all_penalties))
    else:
        model.Add(total_penalty == 0)

    # Preserve prior schedule first, then optimize penalties and balance.
    if keep_vars:
        model.Maximize(sum(keep_vars) * 1000 - total_penalty * 10 - max_load)
    else:
        model.Minimize(total_penalty * 10 + max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    day_order = get_day_order()
    shift_order = {"morning": 1, "evening": 2}
    schedule.sort(
        key=lambda r: (
            day_order.get(r["day"], 999),
            shift_order.get(r["shift"], 999),
            r["role"],
            r["employee_name"],
        )
    )
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    """Generate an initial schedule from scratch."""
    return solve_schedule(data, preferred_schedule=None)


def start_next_week(
    current_data: Dict[str, Any],
    current_final_schedule: List[Dict[str, str]],
) -> Tuple[Dict[str, Any], List[Dict[str, str]]]:
    """Carry final schedule into the next week's initial state."""
    next_week_data = clone_data(current_data)
    next_week_initial_schedule = clone_schedule(current_final_schedule)
    return next_week_data, next_week_initial_schedule


# ---------------------------
# Output and explanation
# ---------------------------

def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    """Print the schedule grouped by day and shift."""
    day_order = get_day_order()
    shift_order = {"morning": 1, "evening": 2}

    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(
        grouped.keys(),
        key=lambda x: (day_order.get(x[0], 999), shift_order.get(x[1], 999))
    ):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def print_preferences(data: Dict[str, Any]) -> None:
    """Print all currently stored preferences."""
    prefs = data.get("preferences", [])
    print("\n===== CURRENT PREFERENCES =====")
    if not prefs:
        print("No preferences.")
    else:
        for i, pref in enumerate(prefs, start=1):
            print(f"{i}. {pref}")
    print("===============================\n")


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    """Print removed and added assignments between two schedules."""
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    day_order = get_day_order()
    shift_order = {"morning": 1, "evening": 2}

    def sort_key(item: Tuple[str, str, str, str]) -> Tuple[int, int, str, str]:
        emp_id, day, shift, role = item
        return (day_order.get(day, 999), shift_order.get(shift, 999), role, emp_id)

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed, key=sort_key):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added, key=sort_key):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def generate_explanation(
    data_before: Dict[str, Any],
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> str:
    """Generate a short, natural, and concise explanation."""
    change_type = change.get("type", "").strip()
    _, added_rows = get_removed_and_added_assignments(old_schedule, new_schedule)

    if change_type == "unavailable":
        resolved_emp = resolve_employee_name(data_before["staff"], change["employee_name"])
        name = resolved_emp["name"]
        day = change["day"]
        shift = change["shift"]

        replacements = [
            r["employee_name"] for r in added_rows
            if r["day"] == day and r["shift"] == shift
        ]

        if replacements:
            rep = ", ".join(sorted(set(replacements)))
            return f"{name} is unavailable for {day} {shift}, so {rep} covers it instead."
        return f"{name} is unavailable for {day} {shift}, so the shift was reassigned."

    if change_type == "direct_swap":
        resolved_emp_1 = resolve_employee_name(data_before["staff"], change["employee_name_1"])
        resolved_emp_2 = resolve_employee_name(data_before["staff"], change["employee_name_2"])
        n1 = resolved_emp_1["name"]
        n2 = resolved_emp_2["name"]
        d1 = change["day_1"]
        s1 = change["shift_1"]
        d2 = change["day_2"]
        s2 = change["shift_2"]

        return f"{n1} and {n2} swapped {d1} {s1} and {d2} {s2}."

    if change_type == "back_to_back_preference":
        resolved_emp = resolve_employee_name(data_before["staff"], change["employee_name"])
        name = resolved_emp["name"]
        penalty = int(change.get("penalty", 5))

        old_same, old_cross = get_back_to_back_counts(old_schedule, name)
        new_same, new_cross = get_back_to_back_counts(new_schedule, name)

        if penalty > 0:
            return f"{name} now has fewer back-to-back shifts ({old_same}/{old_cross} → {new_same}/{new_cross})."
        if penalty < 0:
            return f"{name} now has more back-to-back shifts ({old_same}/{old_cross} → {new_same}/{new_cross})."
        return f"No change to back-to-back shifts for {name}."

    if change_type == "shift_preference":
        resolved_emp = resolve_employee_name(data_before["staff"], change["employee_name"])
        name = resolved_emp["name"]
        day = change.get("day")
        shift = change.get("shift")
        penalty = int(change.get("penalty", 5))

        before = count_matching_assignments(old_schedule, name, day, shift)
        after = count_matching_assignments(new_schedule, name, day, shift)

        if day and shift:
            label = f"{day} {shift}"
        elif day:
            label = day
        elif shift:
            label = shift
        else:
            label = "that pattern"

        if penalty > 0:
            return f"{name} now works fewer {label} shifts ({before} → {after})."
        if penalty < 0:
            return f"{name} now works more {label} shifts ({before} → {after})."
        return f"No change for {name} on {label} ({before})."

    if change_type == "unavailable_shift":
        resolved_emp = resolve_employee_name(data_before["staff"], change["employee_name"])
        name = resolved_emp["name"]
        day = change.get("day")
        shift = change.get("shift")

        before = count_matching_assignments(old_schedule, name, day, shift)
        after = count_matching_assignments(new_schedule, name, day, shift)

        if day and shift:
            label = f"{day} {shift}"
        elif day:
            label = day
        elif shift:
            label = shift
        else:
            label = "that pattern"

        return f"{name} is no longer allowed to work {label} ({before} → {after})."

    return "Schedule updated."


def print_explanation(explanation: str) -> None:
    """Print the generated explanation block."""
    print("===== EXPLANATION =====")
    print(explanation)
    print("=======================\n")


# ---------------------------
# Update logic
# ---------------------------

def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    """Apply one requested change and re-optimize the schedule."""
    change_type = change.get("type", "").strip()
    updated_data = clone_data(data)

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = normalize_day(change["day"])
        shift = normalize_shift(change["shift"])

        emp = resolve_employee_name(data["staff"], employee_name)
        set_availability(updated_data, emp["id"], day, shift, False)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "unavailable_shift":
        employee_name = change["employee_name"].strip()
        target_day = normalize_optional_day(change.get("day"))
        target_shift = normalize_optional_shift(change.get("shift"))

        emp = resolve_employee_name(data["staff"], employee_name)
        set_availability_by_pattern(
            updated_data,
            emp["id"],
            available=False,
            target_day=target_day,
            target_shift=target_shift,
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "direct_swap":
        name_1 = change["employee_name_1"].strip()
        day_1 = normalize_day(change["day_1"])
        shift_1 = normalize_shift(change["shift_1"])

        name_2 = change["employee_name_2"].strip()
        day_2 = normalize_day(change["day_2"])
        shift_2 = normalize_shift(change["shift_2"])

        emp_1 = resolve_employee_name(data["staff"], name_1)
        emp_2 = resolve_employee_name(data["staff"], name_2)

        if emp_1["id"] == emp_2["id"]:
            raise ValueError("shift_direct_swap requires two different employees.")

        roles_1 = get_employee_assignment_roles(existing_schedule, name_1, day_1, shift_1)
        roles_2 = get_employee_assignment_roles(existing_schedule, name_2, day_2, shift_2)

        if not roles_1:
            raise ValueError(f"{name_1} is not currently assigned to {day_1} {shift_1}.")
        if not roles_2:
            raise ValueError(f"{name_2} is not currently assigned to {day_2} {shift_2}.")
        if len(roles_1) != 1 or len(roles_2) != 1:
            raise ValueError("Each employee must have exactly one role in their specified shift.")

        role_1 = roles_1[0]
        role_2 = roles_2[0]

        if role_2 not in set(emp_1["skills"]):
            raise ValueError(
                f"{name_1} cannot take {name_2}'s shift because {name_1} lacks skill '{role_2}'."
            )
        if role_1 not in set(emp_2["skills"]):
            raise ValueError(
                f"{name_2} cannot take {name_1}'s shift because {name_2} lacks skill '{role_1}'."
            )

        # Ensure swapped shifts are allowed in availability before forcing assignments.
        set_availability(updated_data, emp_1["id"], day_2, shift_2, True)
        set_availability(updated_data, emp_2["id"], day_1, shift_1, True)

        forced_assignments = [
            (emp_1["id"], day_2, shift_2, role_2),
            (emp_2["id"], day_1, shift_1, role_1),
        ]

        updated_schedule = solve_schedule(
            updated_data,
            preferred_schedule=existing_schedule,
            forced_assignments=forced_assignments,
        )
        return updated_schedule, updated_data

    if change_type == "back_to_back_preference":
        employee_name = change["employee_name"].strip()
        penalty = clamp_preference_penalty(int(change.get("penalty", 5)))

        resolve_employee_name(data["staff"], employee_name)

        add_preference(
            updated_data,
            {
                "type": "back_to_back_preference",
                "employee_name": employee_name,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "shift_preference":
        employee_name = change["employee_name"].strip()
        target_day = normalize_optional_day(change.get("day"))
        target_shift = normalize_optional_shift(change.get("shift"))
        penalty = clamp_preference_penalty(int(change.get("penalty", 5)))

        resolve_employee_name(data["staff"], employee_name)

        pref: Dict[str, Any] = {
            "type": "shift_preference",
            "employee_name": employee_name,
            "penalty": penalty,
        }
        if target_day:
            pref["day"] = target_day
        if target_shift:
            pref["shift"] = target_shift

        add_preference(updated_data, pref)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'shift_unavailability', 'pattern hard restriction', 'shift_direct_swap', 'back_to_back_preference', and 'shift_preference'."
    )


# ---------------------------
# Interactive input flow
# ---------------------------

def prompt_change_request() -> Dict[str, Any]:
    """Prompt the user for one supported schedule update request."""
    print("\nChoose an update type:")
    print("1. shift_unavailability")
    print("2. shift_direct_swap")
    print("3. back_to_back_preference")
    print("4. shift_preference")

    update_type_aliases = {
        "1": "1",
        "unavailable": "1",
        "shift unavailability": "1",
        "shift_unavailability": "1",
        "not available": "1",
        "unavailable shift": "1",

        "2": "2",
        "directswap": "2",
        "direct swap": "2",
        "shift direct swap": "2",
        "shift_direct_swap": "2",
        "swap": "2",

        "3": "3",
        "backtobackpreference": "3",
        "back to back": "3",
        "back-to-back": "3",
        "back to back preference": "3",
        "back_to_back_preference": "3",

        "4": "4",
        "shiftpreference": "4",
        "shift preference": "4",
        "shift_preference": "4",
        "preference": "4",
        "hard restriction": "4",
        "restriction": "4",
    }

    while True:
        choice_raw = input("Enter 1, 2, 3, or 4: ")
        choice_key = simplify_text(choice_raw)
        choice = update_type_aliases.get(choice_key) or update_type_aliases.get(choice_key.replace(" ", ""))
        if choice in {"1", "2", "3", "4"}:
            break
        print("Please enter 1, 2, 3, or 4.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = normalize_day(input("Day (e.g. Monday/Saturday): ").strip())
        shift = normalize_shift(input("Shift (e.g. morning/evening): ").strip())

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    if choice == "2":
        employee_name_1 = input("Employee 1 name: ").strip()
        day_1 = normalize_day(input("Employee 1 current day: ").strip())
        shift_1 = normalize_shift(input("Employee 1 current shift: ").strip())

        employee_name_2 = input("Employee 2 name: ").strip()
        day_2 = normalize_day(input("Employee 2 current day: ").strip())
        shift_2 = normalize_shift(input("Employee 2 current shift: ").strip())

        return {
            "type": "direct_swap",
            "employee_name_1": employee_name_1,
            "day_1": day_1,
            "shift_1": shift_1,
            "employee_name_2": employee_name_2,
            "day_2": day_2,
            "shift_2": shift_2,
        }

    if choice == "3":
        employee_name = input("Employee name: ").strip()
        print("Back-to-back preference (-10 to 10):")
        print("Positive = avoid, negative = prefer.")

        while True:
            penalty_text = input("Penalty (default 5): ").strip()
            try:
                penalty = int(penalty_text) if penalty_text else 5
                penalty = clamp_preference_penalty(penalty)
                break
            except ValueError as e:
                print(e)

        return {
            "type": "back_to_back_preference",
            "employee_name": employee_name,
            "penalty": penalty,
        }

    employee_name = input("Employee name: ").strip()
    print("Choose preference scope:")
    print("1. a shift type (e.g. evening)")
    print("2. a day (e.g. Tuesday)")
    print("3. a specific day and shift (e.g. Tuesday evening)")

    sub_choice_aliases = {
        "1": "1",
        "shift": "1",
        "shift type": "1",
        "a shift": "1",
        "2": "2",
        "day": "2",
        "a day": "2",
        "3": "3",
        "specific": "3",
        "specific day and shift": "3",
        "day and shift": "3",
    }

    while True:
        sub_choice_raw = input("Enter 1, 2, or 3: ")
        sub_choice_key = simplify_text(sub_choice_raw)
        sub_choice = sub_choice_aliases.get(sub_choice_key) or sub_choice_aliases.get(sub_choice_key.replace(" ", ""))
        if sub_choice in {"1", "2", "3"}:
            break
        print("Please enter 1, 2, or 3.")

    target_day: str | None = None
    target_shift: str | None = None

    if sub_choice == "1":
        target_shift = normalize_shift(input("Shift (e.g. evening): ").strip())
    elif sub_choice == "2":
        target_day = normalize_day(input("Day (e.g. Tuesday): ").strip())
    else:
        target_day = normalize_day(input("Day (e.g. Tuesday): ").strip())
        target_shift = normalize_shift(input("Shift (e.g. evening): ").strip())

    hard_restriction = ask_yes_no("Hard restriction? (y/n): ")
    if hard_restriction:
        change: Dict[str, Any] = {
            "type": "unavailable_shift",
            "employee_name": employee_name,
        }
        if target_day:
            change["day"] = target_day
        if target_shift:
            change["shift"] = target_shift
        return change

    print("Shift preference (-10 to 10):")
    print("positive = avoid, negative = prefer")
    while True:
        penalty_text = input("Penalty (default 5): ").strip()
        try:
            penalty = int(penalty_text) if penalty_text else 5
            penalty = clamp_preference_penalty(penalty)
            break
        except ValueError as e:
            print(e)

    change: Dict[str, Any] = {
        "type": "shift_preference",
        "employee_name": employee_name,
        "penalty": penalty,
    }
    if target_day:
        change["day"] = target_day
    if target_shift:
        change["shift"] = target_shift
    return change


# ---------------------------
# Main program
# ---------------------------

def main() -> None:
    """Run the interactive scheduling workflow."""
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule for Week 1...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    week_number = 1

    while True:
        print_schedule(schedule, title=f"WEEK {week_number} INITIAL SCHEDULE")
        print_preferences(data)

        while True:
            should_update = ask_yes_no(f"Do you want to enter an update for Week {week_number}? (y/n): ")
            if not should_update:
                break

            try:
                change_request = prompt_change_request()
                updated_schedule, updated_data = update_schedule(data, schedule, change_request)

                print_schedule(updated_schedule, title=f"WEEK {week_number} UPDATED SCHEDULE")
                compare_schedules(schedule, updated_schedule)

                explanation = generate_explanation(data, schedule, updated_schedule, change_request)
                print_explanation(explanation)

                print_preferences(updated_data)

                data = updated_data
                schedule = updated_schedule

                save_changes = ask_yes_no("Do you want to save updates back to JSON? (y/n): ")
                if save_changes:
                    save_data_to_json(data, data_file)
                    print(f"Saved updated data to {data_file}")

            except ValueError as e:
                print(f"Error: {e}")

            show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
            if show_current:
                print_schedule(schedule, title=f"WEEK {week_number} CURRENT FINAL SCHEDULE")
                print_preferences(data)

        print_schedule(schedule, title=f"WEEK {week_number} FINAL SCHEDULE")

        move_to_next_week = ask_yes_no(
            f"Do you want to move from Week {week_number} to Week {week_number + 1}? (y/n): "
        )
        if not move_to_next_week:
            print("Done.")
            break

        data, schedule = start_next_week(data, schedule)
        week_number += 1
        print(
            f"Started Week {week_number}. "
            f"By rule, Week {week_number - 1} final schedule is now Week {week_number} initial schedule."
        )


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule for Week 1...

===== WEEK 1 INITIAL SCHEDULE =====

Monday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Monday - evening
  cashier  -> Brian (E02)
  server   -> Eric (E05)
  server   -> George (E07)

Tuesday - morning
  cashier  -> Julia (E10)
  server   -> Daniel (E04)

Tuesday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> George (E07)

Wednesday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Wednesday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Thursday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Thursday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> Eric (E05)

Friday - morning
  cashier  -> Hannah (E08)
  server   -> Alice (E01)

Friday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E0